In [1]:
import numpy as np
import networkx as nx
import xgi
import json
from tqdm import tqdm
from multiprocess import Pool
from itertools import combinations
from math import comb
import random
from scipy.spatial import distance

from netsimile import *
from portrait_divergence import *
from hypergraph_models import *
from hypergraph_null_models import *

# Generate hypergraph models

In [3]:
H_models = {}
n_samp = 100   # number of samples of each model
Nmin, Nmax = 200, 300

# random hypergraphs
for s in range(n_samp):
    random.seed(s)
    N = int(random.uniform(Nmin,Nmax))
    k_avg = np.round(random.uniform(1.5,2.), 1) 
    ks = [k_avg] * 6   # 6 means hyperedge sizes -> s=2,3,4,5,6,7
    H_models[f'ER{s}_{N}_{k_avg}'] = stratified_er_hypergraph(N, ks, p_type='degree', seed=s)

# configuration model hypergraphs 
# with power-law degree distribution 
for s in range(n_samp):
    seed=s+n_samp
    random.seed(seed)
    N = int(random.uniform(Nmin,Nmax))
    gamma = random.uniform(2.,2.05)
    degs = powerlaw_degree_distribution(N, gamma, seed, cutoff=30)
    deg_lists = [degs] * 2   # 2 means hyperedge sizes -> s=2,3
    H_models[f'CM{s}_{N}_{gamma}'] = stratified_cm_hypergraph(deg_lists, seed=seed)
    
# Watts-Strogatz hypergraphs
for s in range(n_samp):
    seed=s+n_samp*2
    random.seed(seed)
    N = int(random.uniform(Nmin,Nmax)) 
    p_rew = np.round(random.uniform(0.15,0.2), 2)
    prs = [p_rew] * 4   # 4 means hyperedge sizes -> s=2,3,4,5
    H_models[f'WS{s}_{N}_{p_rew}'] = stratified_ws_hypergraph(N, prs, seed=seed)

/Users/cosimoagostinelli/Work/xgi-main/xgi/generators/uniform.py:78: UserWarning: This degree sequence is not realizable. Increasing the degree of random nodes so that it is.
  warnings.warn(


# Compute distances between weighted projections

In [5]:
%%time
labels = list(H_models.keys())

# parallelize computations
Gw_list = [weighted_projection(h) for h in H_models.values()]
p = Pool(processes=4)
fvs = p.map(weighted_graph_signature, Gw_list)
paths_G = p.map(weighted_shortest_paths, Gw_list)

# compute distances between all pairs of networks
ds = distance.pdist(np.array(fvs), metric='canberra')
ns_dists = list(ds / len(fvs[0]))
pd_dists = [portrait_divergence_weighted(Gw_list[i], Gw_list[j], paths_G[i], paths_G[j]) 
            for i,j in combinations(range(len(Gw_list)), 2)]

# save results
results = (labels, ns_dists, pd_dists)
with open('../results/NS_PD_weighted_distances_models.json', 'w') as res_file:
    json.dump(results, res_file)

CPU times: user 22min 32s, sys: 2.81 s, total: 22min 34s
Wall time: 23min 7s


# Randomization methods - LH10 data set

In [3]:
with open(f'../data/SocioPatterns/aggr_15min_cliques_thr1_LH10.json') as file: 
    data_ = json.load(file)
# remove eventual edges of lenght 1
data = [i for i in data_ if len(i)>1]
H = xgi.from_hyperedge_list(data)
# relabel and remove eventual multiple edges
H.cleanup(isolates=True, singletons=True, connected=False)
print(H)

Unnamed Hypergraph with 76 nodes and 1102 hyperedges


In [5]:
# number of samples of each null model
n_samp = 50  
 
Gw = weighted_projection(H)   
fvs = [weighted_graph_signature(Gw)]
sps = [weighted_shortest_paths(Gw)]
shuff_methods = [edge_shuffled_hypergraph, dp_edge_shuffled_hypergraph, configuration_model_hypergraph]
all_Gw = [Gw]

for F in tqdm(shuff_methods):  
    H_null = [F(H, seed=i) for i in range(n_samp)] 
    Gw_null = [weighted_projection(h) for h in H_null]
    all_Gw += Gw_null
    # parallelize computations
    p = Pool(processes=5)
    fvs += p.map(weighted_graph_signature, Gw_null)
    sps += p.map(weighted_shortest_paths, Gw_null)

# compute distances between all pairs of networks
ds = distance.pdist(np.array(fvs), metric='canberra')
ns_dists = list(ds / len(fvs[0]))
pd_dists = [portrait_divergence_weighted(all_Gw[i], all_Gw[j], sps[i], sps[j])
            for i,j in combinations(range(len(all_Gw)), 2)]

labels = ['original'] 
for l in ['RS','PS','DS']:
    labels += [f'{l}{i}' for i in range(n_samp)]
    
results = (labels, ns_dists, pd_dists)
    
# save results
with open('../results/reshuffling_NS_PD_weighted_SocioPatterns_LH10.json', 'w') as res_file:
    json.dump(results, res_file)

100%|█████████████████████████████████████████████| 3/3 [02:09<00:00, 43.22s/it]
